# Deep Momentum V2 — research master

This notebook is the entry point you can follow in VS Code and rerun with **Run All**.
It calls small, tested worker files in `src/dmv2/`; the loading rules are coded there.

**Selected design:** research initialization in **2010**, rolling **10 years training / 1 year testing**.
Nominal outcome tests are **2020–2023**. Starting later is a research choice, not a claim that earlier raw history is absent.
The source files still contain their longer histories.

Current Run All performs configuration, fold planning, source checks and one bounded
historical-universe audit, then a bounded monthly-return engineering sample.
Daily prices stay intact for later daily backtesting. A bounded feature stage calculates daily indicators and exact month-end snapshots. Full panels and models require the later execution plan.
No market-capitalization input is used. Protected 2024+ financial values remain outside this workflow.


## 1. Environment and project root

Select **Deep Momentum V2 (.venv)** in the VS Code kernel picker. This bootstrap
finds V2 from its root, the `notebooks/` folder or the Finance-Working-Files repository.
It imports the current local worker code explicitly and enables reloads during development.
It does not install packages, log in, download data or open cloud resources.


In [ ]:
from pathlib import Path
import sys
import json
from dataclasses import asdict
from html import escape
from IPython.display import display, HTML, Markdown

# Clear prior run evidence before setup, so failed reruns cannot display or save old results.
settings = catalog = inspection = audit_path = normalization_protocol = None
identity_audit = identity_audit_path = None
research_identity_df = research_identity_summary = None
feature_run = feature_history_df = features_df = feature_coverage_df = feature_catalog_df = feature_artifact_path = None
monthly_run = daily_df = monthly_df = numerical_preview = monthly_artifact_path = calendar_audit = None

cwd = Path.cwd().resolve()
bases = [cwd, *cwd.parents]
candidates = bases + [p / "Investments_2026" / "deep_momentum_v2" for p in bases]
ROOT = next((p for p in candidates
             if (p / "configs/research_workflow.json").is_file()
             and (p / "src/dmv2/workflow.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the V2 folder in VS Code and restart the notebook kernel.")
SRC = (ROOT / "src").resolve()
for name, module in list(sys.modules.items()):
    if name == "dmv2" or name.startswith("dmv2."):
        location = getattr(module, "__file__", None)
        if location and not Path(location).resolve().is_relative_to(SRC):
            raise RuntimeError("The kernel has V2 code from another folder. Restart the kernel.")
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
import dmv2
assert Path(dmv2.__file__).resolve().is_relative_to(SRC)
get_ipython().run_line_magic("load_ext", "autoreload")
get_ipython().run_line_magic("autoreload", "2")
from dmv2.catalog import SourceCatalog
from dmv2.chronology import build_annual_folds
from dmv2.workflow import (load_workflow, run_inspection, save_inspection,
                            run_monthly_sample, save_monthly_sample)

def show_table(rows, columns):
    header = "".join(f"<th>{escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(r.get(c, '')))}</td>" for c in columns) + "</tr>"
                   for r in rows)
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

print("Project:", ROOT)
print("Python:", sys.executable)
print("Worker code:", Path(dmv2.__file__).parent)

from dmv2.feature_workflow import run_feature_sample, save_feature_sample
from dmv2.features import predictor_frame, training_slice

from dmv2.workflow import workflow_identity_policy
from dmv2.identity_policy import identity_policy_provenance, qualify_audit_ledger


## 2. One settings file

Edit `configs/research_workflow.json`, then rerun this notebook from the top.
The study start, engineering audit date and bounded monthly sample come from this file.
The current initialization is 2010; the engineering audit date is a separate setting.
`engineering_sample = combined` is a loader demonstration, not a frozen primary modelling universe.
Price sampling is off by default; enabling it reads at most the explicitly configured small ID sample.

The monthly stage is enabled for **January–February 2010 and two requested price IDs**.
Its full monthly cohort ledger is retained. A December 2009 signal anchor is necessary for
January returns; earlier observations are feature warmup, not earlier training outcomes.

In [ ]:
settings = None
settings = load_workflow(ROOT)
print(json.dumps(settings, indent=2))
monthly_controls = settings["monthly_returns"]
display(Markdown(f"Research initialization: **{settings['research_start']}**; "
                 f"engineering snapshot: **{settings['inspection']['as_of']}**; "
                 f"monthly engineering sample: **{monthly_controls['start_month']}–{monthly_controls['end_month']}**, "
                 f"**{monthly_controls['sample_size']} requested price IDs**."))

print(identity_policy_provenance(workflow_identity_policy(settings))["warning"])


### Active normalization and missingness rules

`configs/normalization_protocol.json` records the active engineering contract
under our accepted provisional identity policy. The former proposal and its
2000–2009 strict-identity audit are archived reference evidence.

Request every admitted original member before signal-time statistics. The
paper's 15 inputs share one population with all five exact finite raw momenta,
N≥2 and positive finite sample SD at every horizon. Preserve shared meanMOM
and stdMOM levels over time. The two-ID sample withholds normalization because
it is not an exhaustive cohort request.

For added predictors, fit medians and the retained schema only inside the
relevant training partition; add a fixed missingness indicator for each retained
feature. No extra scaling, clipping, winsorization or full-history feature screen
is used. Never impute a label. Missing added features do not remove otherwise
qualified prediction rows. Context features retain their explicit complete-cohort
policy and remain withheld without the required original/daily context.


In [ ]:
normalization_protocol = None
normalization_protocol = json.loads((ROOT / "configs/normalization_protocol.json").read_text())
assert settings["features"]["cohort_policy"] == normalization_protocol["paper_normalization"]["policy"]
assert normalization_protocol["chronology"]["initial_outcome_year"] == int(settings["research_start"][:4])
print("Active protocol:", normalization_protocol["protocol_id"])
print("Status:", normalization_protocol["status"])
print("Paper population policy:", normalization_protocol["paper_normalization"]["policy"])
print("Missingness is fitted only on completed, explicitly selected training rows.")
print("No preprocessing fit or model fit is run here.")


## 3. Rolling chronology — calendar plan, not a trained model

These windows refer to **completed outcome years**. A December 2019 signal
predicting January 2020 belongs to the 2020 test; its future label cannot enter
training ending in December 2019. Actual training rows must also have observable labels at the fit cutoff.

The nominal 2010–2019 → 2020 fold needs verified coverage and feature warmup.
Early 2010 inputs use exact pre-signal windows, including pre-2010 prices.
Selecting 2010 does not truncate the raw files or establish 120 complete usable training months.
Rolling classifier history is our chosen adaptation to the paper's expanding history.
The paper's trailing ten-year class-payoff estimator is a separate component.


In [ ]:
training = settings["training"]
folds = build_annual_folds(settings["research_start"], training["last_test_year"],
                           training["train_years"], mode=training["mode"])
show_table([asdict(f) for f in folds],
           ["fold_id", "train_start", "train_end", "label_cutoff", "test_start", "test_end"])
print(len(folds), "nominal annual folds; no features, fits or backtests executed.")
first_fold, last_fold = folds[0], folds[-1]
display(Markdown(f"First fold: training **{first_fold.train_start}–{first_fold.train_end}**, "
                 f"test **{first_fold.test_start}–{first_fold.test_end}**. "
                 f"Final test ends **{last_fold.test_end}**."))


## 4. Local source inventory

The catalog uses manifest-listed files and canonical manifest pins, not directory discovery.
This checks file presence and sizes. It does not repeat the prior whole-file SHA-256
verification or claim complete historical returns. A fresh clone needs your own approved
`configs/data_sources.local.json` and downloaded source releases.


In [ ]:
catalog = None
catalog = SourceCatalog(ROOT)
inventory = catalog.verify(hashes=False)
show_table([asdict(r) for r in inventory],
           ["source_key", "release_id", "object_count", "size_bytes", "hashes_checked"])
print(sum(r.object_count for r in inventory), "pinned local objects present.")


## 5. Bounded historical constituent audit

The default query is one source session, **29 January 2010**, for the S&P 500/400 union.
The date is read from `inspection.as_of`; this is a separate bounded inspection.
It does not scan all price history or build the research-period feature panel. Source storage is arranged
by security/index history, so many metadata rows may be considered for this one-date result;
the configured row/file/byte limits apply and failures never return truncated success.

`historical_reconstruction` respects supplied date rules. The sources do not certify true
historical knowledge-time identity/revision evidence. Current delisting flags and future
quote availability never select the historical cohort.


In [ ]:
inspection = audit_path = None
inspection = run_inspection(catalog, settings)
if inspection is None:
    display(Markdown("Universe inspection is disabled in the settings file."))
else:
    print("Research initialization:", settings["research_start"])
    print("Engineering snapshot:", inspection.snapshot.as_of)
    print(json.dumps(inspection.snapshot.summary, indent=2))
    show_table(inspection.snapshot.table.select(
        ["source_security_id", "membership_tickers", "indices", "bridge_method",
         "bridge_confidence", "identity_status"]).to_pylist()[:12],
        ["source_security_id", "membership_tickers", "indices", "bridge_method",
         "bridge_confidence", "identity_status"])
    print("Preview only; all", inspection.snapshot.table.num_rows, "constituents stay in the ledger.")


### Read the identity statuses correctly

`needs_review` preserves an upstream caution flag. It does **not** mean the stock is
absent or its price history is necessarily wrong. The prior 2010 sample's 755 flagged
links all had provider IDs and successful resolution; 751 used ticker/company matches.
Some warnings can represent harmless aliases; date-specific conflicts need separate evidence.

The notebook retains all original constituents. Do not train on only the unflagged subset
as though it were the full intended universe. Historical identity validity and terminal-event
accounting remain the next data-quality tasks.


### Historical identity evidence review — reference anchors

This separate metadata worker still checks **2000-01-31, 2005-01-31 and 2009-01-30**.
These fixed diagnostic anchors predate the current 2010 initialization and do not
measure its training coverage. The current engineering snapshot is controlled separately.
 The exact three-date membership filter and shared identity tables have a cumulative ceiling of 20 million candidate rows, 100,000 returned metadata rows, 12 data files and 128 MiB projected compressed bytes. No price or outcome table is read.

Every original member stays in `identity_audit.table` and its private review queue. ISIN aliases are source agreement, not independent provider confirmation. A valid matching declared ISIN query is retrieval-time evidence awaiting raw request provenance. Source identifier dates, even if present, cannot certify undated provider links or publication times. The audit promotes no identities and does not change feature eligibility. See [identity audit method](../docs/IDENTITY_AUDIT.md).


In [ ]:
from dmv2.identity_audit import build_identity_audit, save_identity_audit
identity_audit = identity_audit_path = None
identity_audit = build_identity_audit(catalog)
show_table(identity_audit.summary["snapshots"], [
    "signal_date", "original_members", "original_resolved_members",
    "exact_declared_isin_queries", "dated_identifier_support_rows", "promoted_identities"])
print("Identifier intervals:", identity_audit.interval_profile)
print("Metadata read budget evidence:", identity_audit.provenance["loading_stats"]["total"])
identity_audit_path = save_identity_audit(ROOT, identity_audit)
print("Private identity evidence:", identity_audit_path.relative_to(ROOT))
print("Identity eligibility changed:", identity_audit.summary["eligibility_changed"])


### User-selected provisional identity admission

The original audit remains unchanged. `identity.policy` now permits uniquely linked, resolved provider routes carrying a producer review flag for **provisional research**. Missing/ambiguous routes and same-date provider collisions stay blocked and retain their original rows. `research_identity_df` is a separate admission ledger, never a predictor source.

Original flags are not cleared. Historical identity/PIT/survivorship remain unverified and later research must carry that limitation. Monthly and feature workers consume the same explicit policy. The two-ID smoke selects an unflagged control and a flagged link before prices; unavailable history never triggers a replacement. Full-cohort normalization remains separate from identity admission. See [the recorded decision](../docs/PROVISIONAL_IDENTITY_POLICY.md).


In [ ]:
import pandas as pd
research_identity_df = research_identity_summary = None
admission_rows, research_identity_summary = qualify_audit_ledger(
    identity_audit.table, identity_policy=workflow_identity_policy(settings))
research_identity_df = pd.DataFrame(admission_rows)
show_table(research_identity_summary, ["signal_date", "original_members",
    "identity_eligible_members", "review_flag_accepted_members", "blocked_members"])
print("Admission policy:", workflow_identity_policy(settings))
print("The original identity_audit.table and producer flags remain intact.")


## 6. Optional exact-date price check

Set `inspection.run_price_sample` to `true` for the configured small engineering sample.
This cell summarizes the result. Unqueried members remain `not_requested`; unresolved
identity links remain visible. This does not certify whole-history coverage or returns.


In [ ]:
if inspection is None or inspection.prices is None:
    display(Markdown("This optional one-date quote check is disabled. The separate monthly stage below has its own controls."))
else:
    print(json.dumps(inspection.prices.stats, indent=2))
    print(json.dumps(inspection.quote_coverage.summary, indent=2))


## 7. Private audit artifact and reruns

The complete constituent/exclusion ledger and source/request provenance are saved under
ignored `outputs/notebook_audits/`. The same settings/source/code/runtime pins reuse an identical report.
Worker edits produce a separate fingerprinted artifact; changed evidence under otherwise identical pins raises an error.
The versioned notebook is kept without execution outputs; clear outputs before committing it.


In [ ]:
audit_path = None
if inspection is not None and settings["inspection"]["persist_audit"]:
    audit_path = save_inspection(ROOT, inspection)
    print("Private audit:", audit_path)
else:
    print("Private audit persistence is disabled.")


## 8. Signal-only features before outcomes

The feature worker reads a separate daily history ending at the **latest signal close**.
It never takes `monthly_df` as input. Each earlier snapshot uses only its own past.
`feature_history_df` holds this past-only history; `features_df` contains source-ID/signal
keys and registered predictors. Coverage/identity flags stay in `feature_coverage_df`.

The bank has **728 definitions**: 298 daily technical, 415 S&P 500/400 context and
15 paper monthly momentum/state inputs. Three constants and ten duplicate ranks
remain documented but are excluded by `predictor_frame`. Missing values stay missing.
The strict V2 policy requires exact complete windows and full original cohorts for
cross-sectional statistics. This policy and sample SD (ddof=1) are explicit adaptations.

Daily features are evaluated before selecting exact exchange month ends. The paper
variables retain calendar-month indexing: MOM1 uses month t; MOM3/6/9/12 use
2/5/8/11 completed returns through t-1. Their normalization is at signal time.
The longest abnormal-volume window needs **503 sessions**; observations before the
configured 2010 initialization are warmup rather than earlier training outcomes.

The default two-ID sample cannot support full-universe ranks or paper normalization.
Those outputs stay null with explicit reasons. Market/relative features also stay
null until full dated DAILY membership context is supplied under a measured plan.
No sampled average pretends to be the market.

Features end at signal close t. The next-month return interval is **(t, t+1 month-end]**.
Sharing the signal price as the return's starting anchor does not reveal a future return.
Same-close execution is not assumed. Source revision knowledge, identity continuity and
terminal accounting remain unverified; `knowledge_time_verified` and `model_ready` are false.


In [ ]:
feature_run = feature_history_df = features_df = feature_coverage_df = feature_catalog_df = feature_artifact_path = None
feature_run = run_feature_sample(catalog, settings)
if feature_run is None:
    display(Markdown("Feature engineering is disabled in the settings file."))
else:
    feature_history_df = feature_run.sample.daily_prices.table.to_pandas().copy(deep=True)
    features_df = feature_run.panel.table.to_pandas().copy(deep=True)
    feature_coverage_df = feature_run.panel.coverage.to_pandas().copy(deep=True)
    feature_catalog_df = __import__("pandas").DataFrame(feature_run.panel.specs)
    print(json.dumps(feature_run.panel.summary, indent=2))
    print("Independent feature history request:")
    print(json.dumps(feature_run.sample.loading_provenance["request"], indent=2))
    print("Measured cumulative feature-stage reads:")
    print(json.dumps(feature_run.sample.loading_stats["total"], indent=2))
    assert feature_history_df["trading_date"].max() <= max(feature_run.sample.signal_dates)
    assert "provider_adjusted_return" not in features_df.columns
    assert "return_status" not in features_df.columns


### Inspect stock features and missingness separately

`predictor_frame(feature_run.panel)` is the future X interface. It accepts only
eligible dictionary names and excludes raw momentum diagnostics, outcomes and flags.
It does not fill missing values, fit a scaler or select rows. The display below
is an engineering preview; all original constituents remain in the feature ledger.


In [ ]:
if feature_run is not None:
    preview = features_df.merge(feature_coverage_df[["source_security_id", "signal_date", "feature_status"]],
                                on=["source_security_id", "signal_date"], validate="one_to_one")
    preview = preview.loc[preview["feature_status"] == "partial_features"].head(10)
    show_table(preview.to_dict("records"), ["source_security_id", "signal_date",
        "px_simple_return_20s", "risk_realized_vol_20s", "trend_ma_gap_20s",
        "liq_abnormal_volume_ratio_20s", "zMOM1", "feature_status"])
    print("Dictionary families:")
    display(feature_catalog_df.groupby(["role", "family"], sort=False).size().rename("definitions").to_frame())
    print("Paper raw momentum diagnostics (outside the predictor dictionary):")
    raw = feature_run.panel.diagnostics["paper_raw_momentum"]
    display(raw.loc[raw["MOM1_raw_status"] == "ok"].head(4))
    print("Coverage reasons stay separate from predictors:")
    for name in ("daily", "paper", "context"):
        diagnostic = feature_run.panel.diagnostics[name]
        print(name, "diagnostic rows:", len(diagnostic))
    print("Eligible predictor columns:", predictor_frame(feature_run.panel).shape[1])


In [ ]:
feature_artifact_path = None
if feature_run is not None and settings["features"]["persist"]:
    feature_artifact_path = save_feature_sample(ROOT, feature_run)
    print("Private feature evidence:", feature_artifact_path)
else:
    print("Feature evidence persistence is disabled.")


## 9. Monthly returns with the original historical cohort

Each outcome month uses membership at the **preceding source month end**.
With the current settings, January 2010 uses the December 2009 signal cohort;
February uses the January cohort. The table below derives exact dates from the settings.
A removal during the outcome month remains in that month's ledger. Entrants during
the month enter the next signal cohort, without retroactive admission.

The worker computes `provider_adjusted_return = adj_close[end] / adj_close[signal] - 1`
only at exact consecutive independent XNYS month ends, from completed daily bars and
accepted endpoint quality. No last-observed-price fallback, forward fill, dividend
double counting or zero-return imputation is used. Interior daily gaps are audited
separately from endpoint arithmetic. Unrequested prices are distinct from missing prices.

Source adjustments include splits and dividends, but historical identity continuity,
revision availability, merger stitching and terminal cash flows are not certified.
`accounting_certified`, `knowledge_time_verified` and `daily_backtest_ready` remain false.
These are research return observations, not executable portfolio returns or class labels.

The vendor calendar incorrectly lists some NYSE holidays as sessions. The monthly stage
uses pinned `exchange_calendars` XNYS dates and records reconciliation differences.
It keeps the original vendor calendar; missing quotes never define market holidays.

In [ ]:
monthly_run = daily_df = monthly_df = numerical_preview = monthly_artifact_path = calendar_audit = None
monthly_run = run_monthly_sample(catalog, settings)
if monthly_run is None:
    display(Markdown("Monthly return construction is disabled in the settings file."))
else:
    # Preserve the loaded daily OHLCV/action/quality table. Build a separate monthly view.
    daily_df = monthly_run.sample.daily_prices.table.to_pandas().copy(deep=True)
    monthly_df = monthly_run.sample.panel.table.to_pandas().copy(deep=True)
    print("Daily source observations retained:", len(daily_df))
    print("Original-cohort monthly ledger rows:", len(monthly_df))
    show_table([asdict(w) for w in monthly_run.sample.panel.windows],
               ["outcome_month", "signal_date", "outcome_start", "outcome_end"])
    print(json.dumps(monthly_run.sample.panel.summary, indent=2))
    print("Measured cumulative source reads:")
    print(json.dumps(monthly_run.sample.loading_stats["total"], indent=2))
    print("Calendar reconciliation:")
    calendar_audit = monthly_run.sample.loading_provenance["calendar_reconciliation"]
    print(json.dumps({key: calendar_audit[key] for key in
                      ("policy", "package_version", "scope", "source_only_dates",
                       "independent_only_dates", "independent_session_count")}, indent=2))

### Read `monthly_df` before using its numbers

There is one row for **every source member at each signal cutoff**. `return_status`
separates valid ratios, unrequested observations, unresolved identities, missing
endpoints and rejected prices. A missing end can require terminal-event review; it
is not proof of a delisting and never deletes the constituent. The two-ID sample
does not establish complete price coverage of the intended research universe.

`daily_quote_path_complete` audits the independent XNYS daily sessions, independently of
`daily_backtest_ready`. Numeric month-end arithmetic can exist with an interior
gap; future daily backtests must resolve that path and cash-flow accounting.
Monthly volume sums are present only for complete valid daily observations, and
do not represent share turnover.

The `outcome_month` return becomes observable after its month closes. It must not
be a feature at the earlier `signal_date`. No next-month target ranks are built here.
Future feature construction must also retrieve an entrant's legitimate pre-entry
price history; membership selects prediction cohorts, not a truncation of price history.

In [ ]:
numerical_preview = None
if monthly_df is not None:
    numerical_preview = monthly_df.loc[monthly_df["return_status"] == "ok"].head(10)
    show_table(numerical_preview.to_dict("records"),
               ["source_security_id", "signal_date", "outcome_month", "outcome_end",
                "provider_adjusted_return", "return_status", "daily_quote_path_complete"])
    print("Engineering numerical preview only. Full monthly_df retains all", len(monthly_df), "cohort rows.")
    print("daily_df remains the separate daily-frequency source view for later backtesting.")

## 10. Keep daily and monthly tables independently

`daily_df` contains the original bounded daily quotes; `monthly_df` is a separate
derived cohort/return table. Monthly aggregation never overwrites the daily source.
The Arrow source table also remains unchanged. This sample is not the future full
daily execution dataset, and no backtest is run.

Persisted private evidence contains **`daily.parquet`**, **`monthly.parquet`** and
**`audit.json`** with source, code, request and table hashes, plus both original and
independent calendar tables. Identical reruns reuse
it; changed evidence under unchanged provenance is rejected. Clear notebook outputs
before committing; source observations and generated tables remain outside Git.

In [ ]:
monthly_artifact_path = None
if monthly_run is not None and settings["monthly_returns"]["persist"]:
    monthly_artifact_path = save_monthly_sample(ROOT, monthly_run)
    print("Private daily/monthly evidence:", monthly_artifact_path)
else:
    print("Monthly evidence persistence is disabled.")

## 11. What follows feature engineering

1. Resolve historical identity/revision limitations and economic terminal-event accounting.
2. Establish full-cohort price coverage and daily context under a measured execution plan.
3. Freeze label classes, publication cutoffs, preprocessing, estimator and trial conventions.
4. Fit and evaluate the rolling 10/1 protocol using completed outcomes only.

`training_slice(feature_run.panel, monthly_run.sample.panel, train_start=..., train_end=...,
label_observed_through=...)` is the explicit future X/y alignment interface. It checks
canonical next-month intervals and completed label cutoffs, retains original denominators,
and returns X, y and an audit ledger separately. It never fits a model or drops missing labels.
Do not select feature rows by future label availability. Any imputation/scaling/selection
must later be fit on training data only; no random stock-row split across calendar months.

The notebook does not construct classes, fit estimators or run backtests.
See `docs/FEATURES.md` for formulas, guards and the present coverage limitations.


## 10. Measured coverage after moving initialization to 2010

This checks complete paper inputs across the original historical S&P 500/400
cohorts, not just the two-ID smoke sample. It separates blocked mappings,
missing signal quotes, invalid quotes and incomplete exact monthly lookbacks.
The 2010 outcome year uses December 2009–November 2010 signals. Older raw
history is retained for warmup; changing initialization does not truncate it.

Run All verifies the saved audit and displays it. The explicit endpoint audit
is a separate worker, not an automatic daily warehouse build, model fit or
backtest. Its coverage must not select predictors or fit missingness values for
earlier folds. See [the measured audit](../docs/INITIALIZATION_2010.md).


In [ ]:
from dmv2.start_coverage import read_saved_start_coverage
initialization_report = initialization_yearly_df = None
INITIALIZATION_EVIDENCE_DIRECTORY = ROOT / "outputs/initialization_coverage/2010-v3"
initialization_report = read_saved_start_coverage(catalog, INITIALIZATION_EVIDENCE_DIRECTORY,
    start_year=int(settings["research_start"][:4]), end_year=settings["training"]["last_test_year"])
initialization_yearly_df = pd.read_csv(INITIALIZATION_EVIDENCE_DIRECTORY / "yearly.csv")
print("Verified initialization audit:", INITIALIZATION_EVIDENCE_DIRECTORY)
display(initialization_yearly_df)
print("This is complete paper-input coverage; it does not certify all added predictors or economic return accounting.")


### Reusable preparation helpers, defined without fitting

The future full feature panel and separate outcome ledger must be supplied
explicitly. `fit_mask` uses completed training labels only. Prediction eligibility
uses only the fifteen signal-time paper inputs; missing held-out outcomes never
select prediction rows. Both returned ledgers retain every original key.
The transformed frame keeps non-admitted rows as missing; an estimator must
select the returned fit mask explicitly rather than dropping the original ledger.


In [ ]:
from dmv2.features import training_slice
from dmv2.preprocessing import fit_preprocessor
from dmv2.paper_features import PAPER_FEATURE_NAMES

def prepare_training_inputs(full_feature_panel, separate_outcomes, *, train_start, train_end, label_observed_through):
    train = training_slice(full_feature_panel, separate_outcomes,
        train_start=train_start, train_end=train_end, label_observed_through=label_observed_through)
    paper_qualified = train.X[list(PAPER_FEATURE_NAMES)].notna().all(axis=1)
    fit_mask = paper_qualified & train.ledger["target_available"]
    fitted = fit_preprocessor(train, fit_mask=fit_mask)
    prepared = fitted.transform(train.X, prediction_mask=paper_qualified)
    return train, fit_mask, fitted, prepared

def prepare_prediction_inputs(signal_X, fitted):
    prediction_mask = signal_X[list(PAPER_FEATURE_NAMES)].notna().all(axis=1)
    return fitted.transform(signal_X, prediction_mask=prediction_mask)

print("Preparation helpers loaded. No imputation fit, model or backtest executed.")
